In [ ]:
# LTV Calculation till 5 items with new to old column

!pip install pandas openpyxl

import pandas as pd
from google.colab import files

# ===============================
# 1. FILE UPLOAD
# ===============================
print("Upload cleaned Excel file:")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_excel(file_name)
df.columns = df.columns.str.strip()

print("Columns detected:")
print(df.columns.tolist())


# ===============================
# 2. KEEP SALES ONLY
# ===============================
df = df[df["Invoice Type"].str.lower() == "sale"]


# ===============================
# 3. FIX NUMERIC BILLING
# ===============================
df["Total Billing Value"] = pd.to_numeric(
    df["Total Billing Value"], errors="coerce"
).fillna(0)


# ===============================
# 4. DATE USING ts COLUMN
# ===============================
df["Invoice_Date"] = pd.to_datetime(df["ts"], errors="coerce")


# ===============================
# 5. SORT BY CUSTOMER DATE
# ===============================
df = df.sort_values(["identity", "Invoice_Date"], ascending=[True, False])


# ===============================
# 6. MULTI-ITEM STACKING
# ===============================
def combine_items(row, field):
    vals = []
    for i in range(1, 6):
        col = f"Item{i}_{field}"
        if col in row and pd.notna(row[col]):
            vals.append(str(row[col]))
    return ", ".join(vals)

df["All_Brands"] = df.apply(lambda r: combine_items(r, "Invoice Brand"), axis=1)
df["All_Models"] = df.apply(lambda r: combine_items(r, "Model Number"), axis=1)
df["All_MRP"] = df.apply(lambda r: combine_items(r, "MRP"), axis=1)
df["All_Billing"] = df.apply(lambda r: combine_items(r, "Billing Value"), axis=1)


# ===============================
# 7. ITEM COUNT (NUM PURCHASES)
# ===============================
df["Item_Count"] = df.apply(
    lambda r: sum(pd.notna(r.get(f"Item{i}_Model Number")) for i in range(1, 6)),
    axis=1
)


# ===============================
# 8. MOST FREQUENT STORE LOGIC
# ===============================
def smart_mode(group, col):

    valid = group[col].dropna()

    if valid.empty:
        return ""

    counts = valid.value_counts()

    # repeated store
    if counts.iloc[0] > 1:
        return counts.index[0]

    # all unique → highest billing store
    idx = group["Total Billing Value"].idxmax()
    return group.loc[idx, col]


# ===============================
# 9. CUSTOMER AGGREGATION
# ===============================
def aggregate_customer(group):

    return pd.Series({

        "LTV": group["Total Billing Value"].sum(),
        "Num Purchases": group["Item_Count"].sum(),

        "CUID Number": ", ".join(group["CUID Number"].dropna().astype(str)),
        "CUID Number (Recent)": group["CUID Number"].iloc[0],

        "Last Purchased Store": group["Store Code"].iloc[0],
        "Most Frequent Store": smart_mode(group, "Store Code"),
        "Most Frequent City": smart_mode(group, "Store City"),
        "Most Frequent Zone": smart_mode(group, "Billing Store Zone"),

        "Invoice Brand": ", ".join(group["All_Brands"]),
        "Model Number": ", ".join(group["All_Models"]),
        "MRP": ", ".join(group["All_MRP"]),
        "Billing Value": ", ".join(group["All_Billing"]),
        "Tender Name": ", ".join(group["Tender Name"].dropna().astype(str)),

        "Store Code": ", ".join(group["Store Code"].astype(str)),
        "Store City": ", ".join(group["Store City"].astype(str)),
        "Billing Store Zone": ", ".join(group["Billing Store Zone"].astype(str)),
        "Invoice Numbers": ", ".join(group["Invoice Number"].astype(str)),

        "First Purchase Date": group["Invoice_Date"].min(),
        "Last Purchase Date": group["Invoice_Date"].max()
    })


ltv = df.groupby("identity", dropna=False).apply(aggregate_customer).reset_index()


# ===============================
# 10. DERIVED METRICS
# ===============================
ltv["Avg Purchase Value"] = ltv["LTV"] / ltv["Num Purchases"]

ltv["Customer Lifetime (Days)"] = (
    ltv["Last Purchase Date"] - ltv["First Purchase Date"]
).dt.days

ltv["Months_Since_Last_Purchase"] = (
    pd.Timestamp.today() - ltv["Last Purchase Date"]
).dt.days // 30

ltv["Price Slab"] = pd.cut(
    ltv["LTV"],
    bins=[0, 100000, 300000, 999999999],
    labels=["<1L", "1-3L", ">3L"]
)

ltv.rename(columns={"identity": "Phone"}, inplace=True)
ltv = ltv.sort_values("LTV", ascending=False)


# ===============================
# 11. EXPORT FILE
# ===============================
output_file = "Final_LTV_Output.xlsx"
ltv.to_excel(output_file, index=False)

files.download(output_file)

print("✅ LTV Calculation Completed Successfully")